# Study 2 Behavioral Heterogeneity Across Sessions

This notebook reproduces the analysis used to compare **inter-individual heterogeneity of affective rating trajectories between H1 and H2**.

## Analysis logic

For each affective dimension (**valence** and **fear**):

1. Each participant's continuous rating trajectory is **z-standardized within session** across time.
2. At each time point, the **sample variance across participants** is calculated.
3. Session-level behavioral heterogeneity is defined as the **mean of these timepoint-specific variances across time**.
4. The session contrast is defined as **H1 − H2**.
5. Sampling uncertainty is estimated with a **participant-level paired bootstrap**:
   - participants are resampled with replacement;
   - the **same participant indices are used for H1 and H2** within each bootstrap draw;
   - the H1−H2 heterogeneity difference is recalculated for each draw;
   - the 95% CI is the 2.5th and 97.5th percentiles of the bootstrap distribution.

### Expected input format

Provide one file for each session × affect combination:

- `valence_H1`
- `valence_H2`
- `fear_H1`
- `fear_H2`

Supported formats:
- `.csv`
- `.tsv`
- `.xlsx`
- `.npy`

The expected matrix shape is:

**rows = participants**  
**columns = time points**

If using CSV/TSV/XLSX and the first column contains participant IDs, set `ID_COLUMN` below to that column name.  
The same participants must be present in H1 and H2, in matching order after ID alignment.

> **Important:** Only edit the paths and, if needed, `ID_COLUMN`. The remaining defaults reproduce the analysis described in the manuscript.


In [ ]:
# ============================================================
# USER CONFIGURATION
# ============================================================

from pathlib import Path

# Replace these with your own data paths.
VALENCE_H1_PATH = Path("YOUR_PATH/valence_H1.csv")
VALENCE_H2_PATH = Path("YOUR_PATH/valence_H2.csv")
FEAR_H1_PATH    = Path("YOUR_PATH/fear_H1.csv")
FEAR_H2_PATH    = Path("YOUR_PATH/fear_H2.csv")

# If your tabular files contain a participant-ID column, specify its name.
# Example: ID_COLUMN = "participant_id"
# Otherwise leave as None.
ID_COLUMN = None

# Reproducibility settings
N_BOOTSTRAP = 10_000
RANDOM_SEED = 42

# Sample SD / sample variance, matching the analysis implementation.
DDOF = 1

# Optional output directory.
OUTPUT_DIR = Path("behavioral_heterogeneity_results")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
# ============================================================
# IMPORTS
# ============================================================

import numpy as np
import pandas as pd
from dataclasses import dataclass
from typing import Optional, Tuple

print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)


In [ ]:
# ============================================================
# DATA LOADING
# ============================================================

@dataclass
class RatingData:
    values: np.ndarray
    participant_ids: Optional[np.ndarray] = None


def load_rating_matrix(path: Path, id_column: Optional[str] = None) -> RatingData:
    """
    Load a participant × time rating matrix.

    Supported:
      - CSV
      - TSV
      - XLSX
      - NPY

    For tabular files:
      - rows = participants
      - columns = time points
      - optional participant ID column may be specified via id_column
    """
    path = Path(path)

    if not path.exists():
        raise FileNotFoundError(f"File not found: {path}")

    suffix = path.suffix.lower()

    if suffix == ".npy":
        values = np.load(path)
        if values.ndim != 2:
            raise ValueError(f"{path.name}: expected a 2D array, got shape {values.shape}")
        return RatingData(values=np.asarray(values, dtype=float), participant_ids=None)

    if suffix == ".csv":
        df = pd.read_csv(path)
    elif suffix == ".tsv":
        df = pd.read_csv(path, sep="\t")
    elif suffix in {".xlsx", ".xls"}:
        df = pd.read_excel(path)
    else:
        raise ValueError(
            f"Unsupported file type: {suffix}. "
            "Use .csv, .tsv, .xlsx, or .npy."
        )

    participant_ids = None
    if id_column is not None:
        if id_column not in df.columns:
            raise ValueError(
                f"{path.name}: ID_COLUMN={id_column!r} not found. "
                f"Available columns: {list(df.columns)}"
            )
        participant_ids = df[id_column].astype(str).to_numpy()
        df = df.drop(columns=[id_column])

    # Require all remaining columns to be numeric.
    numeric = df.apply(pd.to_numeric, errors="coerce")
    if numeric.isna().all(axis=None):
        raise ValueError(f"{path.name}: no numeric rating values were found.")

    values = numeric.to_numpy(dtype=float)

    if values.ndim != 2:
        raise ValueError(f"{path.name}: expected a 2D matrix, got shape {values.shape}")

    return RatingData(values=values, participant_ids=participant_ids)


def align_session_pair(
    h1: RatingData,
    h2: RatingData,
    label: str,
) -> Tuple[np.ndarray, np.ndarray]:
    """
    Align H1 and H2 by participant ID when IDs are available.
    Otherwise require the same number of participants and assume matching row order.
    """

    if h1.participant_ids is not None and h2.participant_ids is not None:
        ids1 = pd.Index(h1.participant_ids)
        ids2 = pd.Index(h2.participant_ids)

        if ids1.has_duplicates or ids2.has_duplicates:
            raise ValueError(f"{label}: duplicated participant IDs detected.")

        common = ids1.intersection(ids2)

        if len(common) != len(ids1) or len(common) != len(ids2):
            missing_h2 = ids1.difference(ids2).tolist()
            missing_h1 = ids2.difference(ids1).tolist()
            raise ValueError(
                f"{label}: H1 and H2 participant sets differ. "
                f"Missing from H2: {missing_h2}; missing from H1: {missing_h1}"
            )

        map1 = {pid: i for i, pid in enumerate(h1.participant_ids)}
        map2 = {pid: i for i, pid in enumerate(h2.participant_ids)}

        order = list(h1.participant_ids)
        x1 = h1.values[[map1[pid] for pid in order], :]
        x2 = h2.values[[map2[pid] for pid in order], :]
        return x1, x2

    if h1.values.shape[0] != h2.values.shape[0]:
        raise ValueError(
            f"{label}: H1 and H2 have different participant counts "
            f"({h1.values.shape[0]} vs {h2.values.shape[0]})."
        )

    return h1.values, h2.values


In [ ]:
# ============================================================
# ANALYSIS FUNCTIONS
# ============================================================

def zscore_subjectwise(X: np.ndarray, ddof: int = 1) -> np.ndarray:
    """
    Z-standardize each participant's rating trajectory across time.

    X shape:
        participants × time

    This removes between-participant differences in:
      - mean rating level
      - response-scale variability
    """
    X = np.asarray(X, dtype=float)

    mean = np.nanmean(X, axis=1, keepdims=True)
    sd = np.nanstd(X, axis=1, ddof=ddof, keepdims=True)

    invalid = ~np.isfinite(sd) | (sd <= 0)
    if np.any(invalid):
        bad_rows = np.flatnonzero(invalid[:, 0])
        raise ValueError(
            "At least one participant has zero or undefined temporal SD. "
            f"Participant-row indices: {bad_rows.tolist()}"
        )

    return (X - mean) / sd


def cross_participant_heterogeneity(
    Z: np.ndarray,
    ddof: int = 1,
) -> tuple[float, np.ndarray]:
    """
    Compute sample variance across participants at each time point,
    then average these timepoint-specific variances across time.

    Returns
    -------
    session_mean_variance : float
        Mean variance across time.
    variance_by_timepoint : np.ndarray
        Cross-participant variance at every time point.
    """
    Z = np.asarray(Z, dtype=float)

    variance_by_timepoint = np.nanvar(Z, axis=0, ddof=ddof)

    if np.all(~np.isfinite(variance_by_timepoint)):
        raise ValueError("No valid timepoint-specific variances could be computed.")

    session_mean_variance = float(np.nanmean(variance_by_timepoint))
    return session_mean_variance, variance_by_timepoint


def paired_session_bootstrap(
    H1: np.ndarray,
    H2: np.ndarray,
    n_bootstrap: int = 10_000,
    seed: int = 42,
    ddof: int = 1,
):
    """
    Participant-level paired bootstrap comparing H1 and H2 heterogeneity.

    The SAME resampled participant indices are applied to H1 and H2
    within every bootstrap draw, preserving the repeated-participant structure.
    """

    H1 = np.asarray(H1, dtype=float)
    H2 = np.asarray(H2, dtype=float)

    if H1.shape[0] != H2.shape[0]:
        raise ValueError(
            f"H1 and H2 must contain the same number of participants. "
            f"Got {H1.shape[0]} vs {H2.shape[0]}."
        )

    # Participant-wise standardization is performed separately within each session.
    H1_z = zscore_subjectwise(H1, ddof=ddof)
    H2_z = zscore_subjectwise(H2, ddof=ddof)

    # Observed heterogeneity.
    h1_obs, h1_var_t = cross_participant_heterogeneity(H1_z, ddof=ddof)
    h2_obs, h2_var_t = cross_participant_heterogeneity(H2_z, ddof=ddof)
    diff_obs = h1_obs - h2_obs

    n_subjects = H1_z.shape[0]
    rng = np.random.default_rng(seed)

    boot_h1 = np.empty(n_bootstrap, dtype=float)
    boot_h2 = np.empty(n_bootstrap, dtype=float)
    boot_diff = np.empty(n_bootstrap, dtype=float)

    for b in range(n_bootstrap):
        # Paired participant bootstrap:
        # same participant indices used for H1 and H2.
        idx = rng.choice(n_subjects, size=n_subjects, replace=True)

        h1_b, _ = cross_participant_heterogeneity(H1_z[idx, :], ddof=ddof)
        h2_b, _ = cross_participant_heterogeneity(H2_z[idx, :], ddof=ddof)

        boot_h1[b] = h1_b
        boot_h2[b] = h2_b
        boot_diff[b] = h1_b - h2_b

    ci_low, ci_high = np.quantile(boot_diff, [0.025, 0.975])

    return {
        "H1": h1_obs,
        "H2": h2_obs,
        "H1_minus_H2": diff_obs,
        "CI95_low": float(ci_low),
        "CI95_high": float(ci_high),
        "H1_variance_by_timepoint": h1_var_t,
        "H2_variance_by_timepoint": h2_var_t,
        "bootstrap_H1": boot_h1,
        "bootstrap_H2": boot_h2,
        "bootstrap_difference": boot_diff,
        "n_subjects": n_subjects,
        "n_bootstrap": n_bootstrap,
        "seed": seed,
        "ddof": ddof,
    }


In [ ]:
# ============================================================
# LOAD DATA
# ============================================================

val_h1_raw = load_rating_matrix(VALENCE_H1_PATH, ID_COLUMN)
val_h2_raw = load_rating_matrix(VALENCE_H2_PATH, ID_COLUMN)
fear_h1_raw = load_rating_matrix(FEAR_H1_PATH, ID_COLUMN)
fear_h2_raw = load_rating_matrix(FEAR_H2_PATH, ID_COLUMN)

valence_H1, valence_H2 = align_session_pair(
    val_h1_raw, val_h2_raw, label="Valence"
)
fear_H1, fear_H2 = align_session_pair(
    fear_h1_raw, fear_h2_raw, label="Fear"
)

print("Valence H1 shape:", valence_H1.shape)
print("Valence H2 shape:", valence_H2.shape)
print("Fear H1 shape:", fear_H1.shape)
print("Fear H2 shape:", fear_H2.shape)

if valence_H1.shape[0] != fear_H1.shape[0]:
    print(
        "Warning: valence and fear participant counts differ. "
        "This is allowed for separate analyses, but verify that it is intended."
    )


In [ ]:
# ============================================================
# RUN ANALYSES
# ============================================================

valence_result = paired_session_bootstrap(
    valence_H1,
    valence_H2,
    n_bootstrap=N_BOOTSTRAP,
    seed=RANDOM_SEED,
    ddof=DDOF,
)

fear_result = paired_session_bootstrap(
    fear_H1,
    fear_H2,
    n_bootstrap=N_BOOTSTRAP,
    seed=RANDOM_SEED,
    ddof=DDOF,
)


In [ ]:
# ============================================================
# RESULTS SUMMARY
# ============================================================

summary = pd.DataFrame([
    {
        "affect": "valence",
        "H1_variance": valence_result["H1"],
        "H2_variance": valence_result["H2"],
        "H1_minus_H2": valence_result["H1_minus_H2"],
        "CI95_low": valence_result["CI95_low"],
        "CI95_high": valence_result["CI95_high"],
        "n_subjects": valence_result["n_subjects"],
        "n_bootstrap": valence_result["n_bootstrap"],
    },
    {
        "affect": "fear",
        "H1_variance": fear_result["H1"],
        "H2_variance": fear_result["H2"],
        "H1_minus_H2": fear_result["H1_minus_H2"],
        "CI95_low": fear_result["CI95_low"],
        "CI95_high": fear_result["CI95_high"],
        "n_subjects": fear_result["n_subjects"],
        "n_bootstrap": fear_result["n_bootstrap"],
    },
])

display(summary.round(6))

for row in summary.itertuples():
    print(
        f"{row.affect.capitalize()}: "
        f"H1={row.H1_variance:.3f}, "
        f"H2={row.H2_variance:.3f}; "
        f"H1-H2={row.H1_minus_H2:.3f}, "
        f"95% paired-bootstrap CI "
        f"[{row.CI95_low:.3f}, {row.CI95_high:.3f}]"
    )


In [ ]:
# ============================================================
# SAVE RESULTS
# ============================================================

summary_path = OUTPUT_DIR / "behavioral_heterogeneity_summary.csv"
summary.to_csv(summary_path, index=False)

np.savez_compressed(
    OUTPUT_DIR / "behavioral_heterogeneity_bootstrap_draws.npz",
    valence_bootstrap_H1=valence_result["bootstrap_H1"],
    valence_bootstrap_H2=valence_result["bootstrap_H2"],
    valence_bootstrap_difference=valence_result["bootstrap_difference"],
    fear_bootstrap_H1=fear_result["bootstrap_H1"],
    fear_bootstrap_H2=fear_result["bootstrap_H2"],
    fear_bootstrap_difference=fear_result["bootstrap_difference"],
)

np.savez_compressed(
    OUTPUT_DIR / "behavioral_heterogeneity_timepoint_variances.npz",
    valence_H1_variance_by_timepoint=valence_result["H1_variance_by_timepoint"],
    valence_H2_variance_by_timepoint=valence_result["H2_variance_by_timepoint"],
    fear_H1_variance_by_timepoint=fear_result["H1_variance_by_timepoint"],
    fear_H2_variance_by_timepoint=fear_result["H2_variance_by_timepoint"],
)

print("Saved:")
print(" -", summary_path)
print(" -", OUTPUT_DIR / "behavioral_heterogeneity_bootstrap_draws.npz")
print(" -", OUTPUT_DIR / "behavioral_heterogeneity_timepoint_variances.npz")


## Methods text corresponding to this notebook

> **Behavioral heterogeneity across sessions.** To examine whether the affective trajectories differed in inter-individual heterogeneity between the two Study 2 sessions, continuous rating trajectories were analyzed separately for valence and fear. For each affective dimension, each participant’s rating time series was z-standardized separately within H1 and H2 using that participant’s temporal mean and standard deviation. This normalization removed individual differences in overall rating level and response-scale variability, allowing the analysis to focus on cross-participant dispersion in relative temporal response profiles. At each time point, variance was then calculated across participants on the standardized ratings, and session-level heterogeneity was defined as the mean of these timepoint-specific variance estimates across the full rating trajectory.
>
> The session difference was defined as the H1 minus H2 heterogeneity estimate. Sampling uncertainty was assessed using a participant-level paired bootstrap with 10,000 resamples. For each bootstrap draw, participants were sampled with replacement, with the same resampled participant indices applied to H1 and H2 to preserve the repeated-participant structure. Cross-participant variance was recalculated at each time point for each session, averaged across time, and the resulting H1−H2 difference was retained. The 95% confidence interval for the session difference was defined by the 2.5th and 97.5th percentiles of the paired-bootstrap distribution.
